### Using Hugging Face models in Fabric
#### From baseline inference to fine-tuning

#### Session objectives
- Understand how Hugging Face text classification models work in practice
- Build and evaluate a baseline using a pre-trained model
- Interpret model outputs
- Understand *how* fine-tuning works
- Compare baseline vs fine-tuned behaviour and know when fine-tuning is justified


### Environment set up

You can download libraries here, or create an environment and load it. I will show you how to create the environment, however, publishing takes a while, I've already created one.


In [1]:
# !pip install evaluate

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 5, Finished, Available, Finished)

### Problem 

We are solving a supervised text classification problem.

Each input is:
- A short piece of text (e.g. a query, description, or message)

Each output is:
- A discrete label representing a category or outcome

This mirrors many real use cases:
- Customer queries
- Support tickets
- Claims descriptions
- Free-text form inputs

In [2]:
from datasets import load_dataset

dataset = load_dataset("banking77")

dataset

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 6, Finished, Available, Finished)

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

- Hugging Face dataset are versioned, documented and reproducible
- `train` and `test` splits already exist

### Why this could be challenging

Before touching any models, it is important to understand the data:

- Labels may be imbalanced
- Text can be ambiguous or short
- Some labels overlap semantically
- There may be noise or inconsistent labelling

This is exactly the scenario where pre-trained models *sometimes* work well and sometimes fail quietly.

In [3]:
display(dataset["train"])

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5e2f5cd7-c0cd-4ae0-8625-98c9567e8b32)

In [4]:
# Quick inspect of the data and returns label = 11
print(dataset["train"][0])

# Now, decode what label 11 meaning
label_names = dataset["train"].features["label"].names
label_names[11]

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 8, Finished, Available, Finished)

{'text': 'I am still waiting on my card?', 'label': 11}


'card_arrival'

In [5]:
# We can check the count in each label
from collections import Counter

label_counts = Counter(dataset["train"]["label"])
len(label_counts), list(label_counts.items())[:10]
# map each ID to names
for label_id, count in label_counts.items():
    print(label_names[label_id], count)

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 9, Finished, Available, Finished)

card_arrival 153
card_linking 139
exchange_rate 112
card_payment_wrong_exchange_rate 167
extra_charge_on_statement 166
pending_cash_withdrawal 143
fiat_currency_support 126
card_delivery_estimate 112
automatic_top_up 127
card_not_working 112
exchange_via_app 118
lost_or_stolen_card 82
age_limit 110
pin_blocked 115
contactless_not_working 35
top_up_by_bank_transfer_charge 111
pending_top_up 149
cancel_transfer 157
top_up_limits 97
wrong_amount_of_cash_received 180
card_payment_fee_charged 187
transfer_not_received_by_recipient 171
supported_cards_and_currencies 129
getting_virtual_card 98
card_acceptance 59
top_up_reverted 146
balance_not_updated_after_cheque_or_cash_deposit 181
card_payment_not_recognised 168
edit_personal_details 121
why_verify_identity 121
unable_to_verify_identity 102
get_physical_card 106
visa_or_mastercard 135
topping_up_by_card 103
disposable_card_limits 121
compromised_card 86
atm_support 87
direct_debit_payment_not_recognised 182
passcode_forgotten 105
declined

Model do not understand numbers, labels are IDs mapped to meaning.

### Convert to Lakehouse

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

train_df = spark.createDataFrame(dataset["train"])
test_df = spark.createDataFrame(dataset["test"])

train_df.write.format("delta").mode("overwrite").saveAsTable("banking77_train")
test_df.write.format("delta").mode("overwrite").saveAsTable("banking77_test")

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 10, Finished, Available, Finished)

### Baseline: Pre-trained model

We start with a **baseline**:
- No training
- No fine-tuning
- Just inference using a general-purpose language model

Is an off-the-shelf model already good enough?

We choose a well-known generic model `distiltbert-base-uncased`

**DistilBERT** is a compressed version of BERT designed to be smaller, faster, and more efficient, while retaining most of BERT's language understanding capability. It's created using knowlege distillation, where a large "teacher" model (BERT) trains a smaller "student" model (DistilBERT) to mimic its behaviour.

DistilBERT base uncased is commonly used for:

- Text classification
- Sentiment / category detection
- Named Entity Recognition (NER)
- Baseline models before fine-tuning

In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Base (not fine-tuned) model
base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=77 # our data has 77 labels
)
base_tokeniser = AutoTokenizer.from_pretrained("distilbert-base-uncased")

baseline = pipeline(
    "text-classification",
    model=base_model,
    tokenizer=base_tokeniser,
    top_k=3
)

baseline("I am still waiting for my card")


StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 11, Finished, Available, Finished)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[[{'label': 'LABEL_53', 'score': 0.01758783683180809},
  {'label': 'LABEL_45', 'score': 0.016661224886775017},
  {'label': 'LABEL_48', 'score': 0.016260197386145592}]]

#### What does `pipeline()` actually do?

When we call a Hugging Face pipeline, several steps happen automatically:

1. Text is tokenised into numerical IDs
2. Tokens are passed through the transformer encoder
3. The classification head produces raw scores (logits)
4. A softmax converts scores into probabilities
5. The highest-probability label is returned

The pipeline hides this complexity, which is useful but also dangerous if we do not understand the output.


#### Understanding the output

We can see see outputs like:

LABEL_60: 0.15  
LABEL_57: 0.15  

Important points:
- These are probabilities, not confidence
- The model is often unsure
- Small differences still lead to hard decisions
- LABEL names come from training metadata, not intuition

#### Why do all labels have ~15% probability?

The pre-trained model has not been trained on our label set. Because of that, it has no strong signal to prefer one class over another.
So it tells us:
- Model is uncertain
- Base line has little predictive power
- Fine-tuning is likely required for this task.

#### Precision vs Recall
- Precision: When the model predicts X, how often is it correct?
- Recall: Of all true X cases, how many did we catch? 

Different business problems prioritise different trade-offs.

### What fine-tuning does?
Fine-tuning does
- Define real labels
- Trains the decision layer
- Shaprens decision boundaries

### Prepare the data for training

###### Tokenisation


In [8]:
from transformers import AutoTokenizer

tokeniser = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokeniser(
        batch["text"],
        truncation = True, # If sentence is longer than the model can handle, cut it
        padding = "max_length", # Every input is padded to the same length
        max_length = 128
    )

tokenised = dataset.map(tokenize, batched=True)

tokenised.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 12, Finished, Available, Finished)

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Each model has its own tokeniser
The tokeniser defines.. how text is split, which numbers represent which tokens, which special tokens exist.
If you use wrong tokenizer what happens?

- The numbers would not match the models' vocabulary
- The model would receive 'nonsense'
- Training would silently fail.

**So, it is import to match model and tokeniser**

`tokenize` function describes how raw text should be converted into numbers. This applies to the entire dataset.

`padding = "max_length"` is needed as neural networks operate on fixed sized tensors so give pad if sentence is shorten than max_length. In our example, we keep each sentence exactly 128 tokens. Short sentence are padded and long sentences are truncated. 



##### Example: how text becomes numbers

Below is a small example showing how a sentence is converted into tokens and token IDs. Each token ID corresponds to an entry in the model's vocabulary.
Special tokens such as `[CLS]`, `[SEP]`, and `[PAD]` are added so the model knows where the sentence starts, ends, and which parts to ignore.

In [9]:
example_text = "I am still waiting for my card"

encoded = tokeniser(
    example_text,
    truncation=True,
    padding="max_length",
    max_length=16,  
    return_tensors=None
)

encoded
# what are input_ids?

print(encoded["input_ids"])
# convert token_ids to tokens to show
tokens = tokeniser.convert_ids_to_tokens(encoded["input_ids"])
# for example, [CLS] token is a special token added to the start of input sequences. Model uses [CLS] token as the summary of the sentence.
print(tokens)
#convert tokens back to text
print(tokeniser.convert_tokens_to_string(tokens))

# Attention mask tells the model which tokens are real text and which are just padding.

print(encoded["attention_mask"])
# 1 - real token (model shoud pay attention)
# 0 - padding (model should ignore)



StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 13, Finished, Available, Finished)

[101, 1045, 2572, 2145, 3403, 2005, 2026, 4003, 102, 0, 0, 0, 0, 0, 0, 0]
['[CLS]', 'i', 'am', 'still', 'waiting', 'for', 'my', 'card', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
[CLS] i am still waiting for my card [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]


- Text to tokens to token IDs
- Token IDs map directly to vocabulary entries
- The tokenizer defines that mapping
- Our model understands only the numbers


### Create a model that is ready to learn our labels

So far, we preparted the data. Now we create a version of the model that is capable of learning our classification task.

At this stage:
- Model understand langugae
- The decision layer is new ans untrained
- The model does not yet know what our labels mean



In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names)
)

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 14, Finished, Available, Finished)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


why we see this warning message?

`Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.`

Language weights are loaded, decisions weights are new and random. So it is saying, I understand langugues, but I don't know how to decide yet.





### Train the model (fine-tuning)



Up to now, our model can read text (tokenisation), it has a decision layer (classifier head), but decision layer is random.

What is training meaning?
When we say "train the model": Show the model many examples of text and correct label, and adjust its decision layer so it makes better choices.

During training, the model repeatedley sees small batches of data. For each batch: 
- Text is tokenised into numerical inputs
- The model produces predictions
- A loss value measures how wrong the predictions are
- The model updates its internal weights to reduce that loss

This process repeats across:
- All batches
- All epochs

Most learning happens in the classification head. The language layers change slowly to preserve general language knowledge.


We are not running this code chunk as it took aver an hour.

In [23]:
from transformers import Trainer, TrainingArguments
import evaluate
import os

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# Where Trainer writes checkpoints
training_dir = "/lakehouse/default/Files/models/banking77/training"
os.makedirs(training_dir, exist_ok=True)

args = TrainingArguments(
    output_dir=training_dir, # where training aftefacts are written during training
    evaluation_strategy="epoch", # evaluate after each full pass. This allows you to see whether the model is improving, whether it is overfitting
    save_strategy="epoch",      # Save a checkpoint after each epoch        
    per_device_train_batch_size=8, # how many examples the model sees before updating itself. (learn from small groups instread of one example at a time). This model learns from 8 examples at a time.
    per_device_eval_batch_size=8, # how many examples are processed toghther during evaluation.
    num_train_epochs=3, # essentially, see every example three times. (here, if you have too many passes, it can cause overfitting - will becomes very good at the training set but worse at new, unseen data) Like training too long teaches the model the answers, not the problem.
    learning_rate=2e-5, # large learning rate: big jumps, small learning rate: careful steps. We want to adjust gentry and avoid destroying learned language knowledge.
    report_to="none"
)

# Trainer orchestrates the entire training loop. 
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenised["train"],
    eval_dataset=tokenised["test"],
    tokenizer=tokeniser,
    compute_metrics=compute_metrics
)

trainer.train()


StatementMeta(, 91f6c4d3-10ec-4292-a6f7-a2e13a6a3251, 27, Finished, Available, Finished)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False)
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,2.256800,1.384662,0.749026
2,0.686200,0.636281,0.868182
3,0.451600,0.499334,0.890260


Checkpoint destination directory /lakehouse/default/Files/models/banking77/training/checkpoint-1251 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory /lakehouse/default/Files/models/banking77/training/checkpoint-2502 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory /lakehouse/default/Files/models/banking77/training/checkpoint-3753 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=3753, training_loss=1.3499484083794162, metrics={'train_runtime': 6992.42, 'train_samples_per_second': 4.292, 'train_steps_per_second': 0.537, 'total_flos': 995132772216576.0, 'train_loss': 1.3499484083794162, 'epoch': 3.0})

We have 10,003 training examples , but the model learns in batches of 8. That gives about 1,251 steps per epoch. Over 3 pochs, that's 3,753 training steps which is waht the progress bar shows. 

#### Interpreting training metrics

- **Epoch**: one full pass through the training data
- **Training loss**: how wrong the model is on the data it learns from
- **Validation loss**: how wrong the model is on unseen data
- **Accuracy**: how often predictions are correct

A healthy training process shows training loss decreasing, validation loss
staying stable or decreasing, and accuracy increasing. If validation loss
starts to rise while training loss keeps falling, the model may be overfitting.


It loads a accuracy metric and calculate to know "what fraction of predictions were correct?"

Model produces scores for every label. We pick the label with the highest score. and we compare this to the true lable, then compute accuracy.

After `trainer.train()` finishes:
- The 'model' object in memoery contains updated (fine-tuned) weights
- The model is now task-aware
- Predictions are no longer guesses
- Probability scores reflect learnd patterns, not uniform uncertainty.

We have updated leared weights.

`compute_metrics()` what it does?

During training, the model outputs **logits**, not labels.

Logits are raw scores that:
- Are not probabilities
- Are not human-interpretable

To evaluate performance, we must:
- Convert logits into predicted labels
- Compare predictions to ground truth labels
- Compute meaningful metrics

`eval_pred` is a tuple containing:
- `logits`: model outputs for each class
- `labels`: the true labels from the dataset

example: logits [[1.2, 0.3, -0.8]
                 [0.1, 2.4, -1.1]]
         labels = [0, 1]
- `argmax` select the index of the highest score and this converts scores into a class label so [1.2, 0.3, -0.8] -> 0,  [0.1, 2.4, -1.1] -> 1 This is model's decision rule.


In [24]:
final_model_dir = "/lakehouse/default/Files/models/banking77_final"
os.makedirs(final_model_dir, exist_ok=True)

model.save_pretrained(final_model_dir)
tokeniser.save_pretrained(final_model_dir)

StatementMeta(, 91f6c4d3-10ec-4292-a6f7-a2e13a6a3251, 28, Finished, Available, Finished)

('/lakehouse/default/Files/models/banking77_final/tokenizer_config.json',
 '/lakehouse/default/Files/models/banking77_final/special_tokens_map.json',
 '/lakehouse/default/Files/models/banking77_final/vocab.txt',
 '/lakehouse/default/Files/models/banking77_final/added_tokens.json',
 '/lakehouse/default/Files/models/banking77_final/tokenizer.json')

`tokenizer_config.json`: This file defines how the tokenizer behaves.
It ensures text is processed exactly the same way every time. 

`special_tokens_map.json`: This file maps special tokens to their meanings. The model was trained expecting these tokens in specific positions. Special tokens tell the model how to read the sentence structure

`vocab.txt`: This is the core vocabulary used by the tokenizer. Token IDs must align exactly with what the model learned. if it is changed, it breaks the link between text and learned weights

`added_tokens.json`: This file records any tokens added beyond the original vocabulary. For example, domain specific terms, custom placeholders. This makes it possible to be domain specific fine tuning or custom business terminology

`tokenizer.json`: This is the fully compiled tokeniser definition. It contains, vocabulary, tokenisation rules, pre/post processing steps. 



In [11]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

load_dir = "/lakehouse/default/Files/models/banking77_final"

model = AutoModelForSequenceClassification.from_pretrained(load_dir)
tokeniser = AutoTokenizer.from_pretrained(load_dir)

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 15, Finished, Available, Finished)

so here, `tokeniser` is reconstructed exactly as it was during the fine-tuning.

In [12]:
# Build mappings
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

# Attach to model config
model.config.id2label = id2label
model.config.label2id = label2id

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 16, Finished, Available, Finished)

In [13]:
# model.config.label2id


StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 17, Finished, Available, Finished)

In [14]:
from transformers import pipeline

finetuned = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokeniser,
    top_k=3
)

finetuned("Why has my withdrawal not posted?")


StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 18, Finished, Available, Finished)

[[{'label': 'pending_cash_withdrawal', 'score': 0.7249761819839478},
  {'label': 'cash_withdrawal_not_recognised', 'score': 0.17270827293395996},
  {'label': 'declined_cash_withdrawal', 'score': 0.025037040933966637}]]

`model=model`
- This is the fine-tuned model object
- It contains updated weights from training
- The classifier head is now task-aware

`tokenizer=tokeniser`
- This tokenizer was loaded from the same directory
- It uses `tokenizer.json` if present (which is as we discussed earlier)
- Text is converted into token IDs exactly as during training

`pipeline(...)`
- Combines tokenizer + model into a single inference object
- Ensures the same preprocessing and label mapping are used


In [15]:
# compare before and after
sample_query = "My card is lost and I need a replacement"
baseline_result = baseline(sample_query)

print(f"Query: {sample_query}")
print(f"Raw Model Prediction: {baseline_result[0][0]['label']} (Confidence: {baseline_result[0][0]['score']:.2f})")
# 'LABEL_X' has no meaning.

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 19, Finished, Available, Finished)

Query: My card is lost and I need a replacement
Raw Model Prediction: LABEL_53 (Confidence: 0.02)


In [16]:
from transformers import pipeline

# BEFORE fine-tuning (untrained classifier head)
baseline_pipeline = pipeline(
    "text-classification",
    model=base_model,
    tokenizer=base_tokeniser
)

# AFTER fine-tuning (our trained model)
finetuned_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokeniser
  
)

StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 20, Finished, Available, Finished)

Baseline pipeline:
- Pre-trained weights
- Generic language understanding
- Weak or uniform predictions

Fine-tuned pipeline:
- Updated weights
- Domain-specific understanding
- Meaningful probability distribution

In [17]:
import pandas as pd

test_queries = [
    "I am still waiting for my card",
    "How do I change my PIN?",
    "Why was my transaction declined?"
    ]

results = []

for q in test_queries:
    # BEFORE fine-tuning
    baseline_out = baseline_pipeline(q)[0]
    
    # AFTER fine-tuning
    finetuned_out = finetuned_pipeline(q)[0]

    results.append({
        "Query": q,
        "Before Fine-Tuning (Label)": baseline_out["label"],
        "Before Fine-Tuning (Score)": round(baseline_out["score"], 3),
        "After Fine-Tuning (Label)": finetuned_out["label"],
        "After Fine-Tuning (Score)": round(finetuned_out["score"], 3),
    })

comparison_df = pd.DataFrame(results)
comparison_df


StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 21, Finished, Available, Finished)

,Query,Before Fine-Tuning (Label),Before Fine-Tuning (Score),After Fine-Tuning (Label),After Fine-Tuning (Score)
0,I am still waiting for my card,LABEL_53,0.018,card_arrival,0.771
1,How do I change my PIN?,LABEL_53,0.018,change_pin,0.906
2,Why was my transaction declined?,LABEL_53,0.018,declined_transfer,0.767


Let's check with our test dataset


In [18]:
def compute_pipeline_accuracy(pipeline, texts, labels):
    correct = 0
    
    for text, true_label in zip(texts, labels):
        pred = pipeline(text)[0]["label"]
        if pred == true_label:
            correct += 1
            
    return correct / len(labels)


StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 22, Finished, Available, Finished)

In [19]:
test_texts = dataset["test"]["text"]
test_labels = [id2label[l] for l in dataset["test"]["label"]]

baseline_accuracy = compute_pipeline_accuracy(
    baseline_pipeline,
    test_texts,
    test_labels
)

finetuned_accuracy = compute_pipeline_accuracy(
    finetuned_pipeline,
    test_texts,
    test_labels
)


StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 23, Finished, Available, Finished)

In [20]:
import pandas as pd

pd.DataFrame([
    {
        "Model": "Baseline (Pre-trained)",
        "Accuracy": round(baseline_accuracy, 3)
    },
    {
        "Model": "Fine-tuned",
        "Accuracy": round(finetuned_accuracy, 3)
    }
])


StatementMeta(, 4a49ff4a-4889-44d6-8503-841ca7c24c25, 24, Finished, Available, Finished)

,Model,Accuracy
0,Baseline (Pre-trained),0.00
1,Fine-tuned,0.89


#### Baseline accuracy ~0.000?

The pre-trained model:
- Has never seen the Banking77 labels
- Does not know how language maps to these intents
- Produces near-uniform probabilities across classes


#### Why is fine-tuned accuracy ~0.90?

After fine-tuning:
- The model has learned how Banking77 intents map to language patterns
- The classifier head is task-aware
- Predictions are no longer guesses


This dataset was designed to highlight the effect of fine-tuning:
- Generic language understanding alone is insufficient
- Task-specific alignment makes a dramatic difference


### Important caveat

High accuracy here does **not** mean:
- Fine-tuning always yields 90% accuracy
- Real-world enterprise data behaves the same way

Production data is usually:
- Noisier
- Less balanced
- More ambiguous

Fine-tuning turns a guessing model into a task-aware model 